# AquaHub — Exploração dos dados CHELSA

## Primeiro protótipo

Objetivo: analisar a temperatura média climatológica (`tas`) do município de Vila Real.

### Períodos de interesse
- 1981–2010 — período histórico
- 2041–2070 — projeção futura
- 2071–2100 — projeção futura

### Fluxo inicial
CHELSA → raster climático → limite municipal → zonal statistics → média municipal → comparação entre períodos.

In [ ]:
import pandas as pd
import geopandas as gpd
import rasterio
import rioxarray as rxr
from exactextract import exact_extract
import matplotlib.pyplot as plt

print("Ambiente AquaHub pronto!")

In [ ]:
from pathlib import Path

print("Diretório atual:")
print(Path.cwd())

In [ ]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

CHELSA_DIR = PROJECT_ROOT / "data" / "raw" / "chelsa"
BOUNDARIES_DIR = PROJECT_ROOT / "data" / "raw" / "boundaries"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Notebook:", NOTEBOOK_DIR)
print("Projeto:", PROJECT_ROOT)
print("CHELSA:", CHELSA_DIR)

In [ ]:
from urllib.request import urlretrieve

filename = "CHELSA_tas_01_1981-2010_V.2.1.tif"

url = (
    "https://os.unil.cloud.switch.ch/"
    "chelsa02/chelsa/global/climatologies/tas/1981-2010/"
    f"{filename}"
)

output_path = CHELSA_DIR / filename

print("Arquivo será salvo em:")
print(output_path)

In [ ]:
if output_path.exists():
    print("Arquivo já existe:", output_path)
else:
    print("Baixando CHELSA...")
    urlretrieve(url, output_path)
    print("Download concluído!")

print(f"Tamanho: {output_path.stat().st_size / 1024**2:.2f} MB")

In [ ]:
import rasterio

with rasterio.open(output_path) as src:
    print("CRS:", src.crs)
    print("Largura:", src.width)
    print("Altura:", src.height)
    print("Número de bandas:", src.count)
    print("Tipo dos dados:", src.dtypes)
    print("NoData:", src.nodata)
    print("Transformação:", src.transform)
    print("Limites:", src.bounds)

In [ ]:
with rasterio.open(output_path) as src:
    print("Scale:", src.scales)
    print("Offset:", src.offsets)
    print("Unidades:", src.units)
    print("Descrição das bandas:", src.descriptions)

    print("\nTags gerais:")
    print(src.tags())

    print("\nTags da banda 1:")
    print(src.tags(1))


In [ ]:
longitude = -7.74
latitude = 41.30

with rasterio.open(output_path) as src:
    raw_value = next(src.sample([(longitude, latitude)]))[0]

    scale = src.scales[0]
    offset = src.offsets[0]

    kelvin = raw_value * scale + offset
    celsius = kelvin - 273.15

print("Valor bruto:", raw_value)
print(f"Temperatura em Kelvin: {kelvin:.2f} K")
print(f"Temperatura em Celsius: {celsius:.2f} °C")

In [ ]:
import geopandas as gpd

caop_path = BOUNDARIES_DIR / "Continente_CAOP2025.gpkg"

print("Arquivo CAOP:")
print(caop_path)

print("\nCamadas disponíveis:")
layers = gpd.list_layers(caop_path)

print(layers)

In [ ]:
municipios = gpd.read_file(
   caop_path,
    layer="cont_municipios"
)

print(municipios.head())
print("\nColunas:")
print(municipios.columns.tolist())

print("\nCRS:")
print(municipios.crs)

In [ ]:
vila_real = municipios[
    municipios["municipio"].str.strip().str.casefold() == "vila real".casefold()
].copy()

print(vila_real)
print("\nQuantidade de registros encontrados:", len(vila_real))

In [ ]:
print("Município:", vila_real.iloc[0]["municipio"])
print("Distrito:", vila_real.iloc[0]["distrito_ilha"])
print("NUTS III:", vila_real.iloc[0]["nuts3"])
print("Área oficial (ha):", vila_real.iloc[0]["area_ha"])
print("Número de freguesias:", vila_real.iloc[0]["n_freguesias"])
print("CRS atual:", vila_real.crs)

In [ ]:
vila_real_4326 = vila_real.to_crs(epsg=4326)

print("CRS original:", vila_real.crs)
print("CRS reprojetado:", vila_real_4326.crs)

print("\nLimites de Vila Real em latitude/longitude:")
print(vila_real_4326.total_bounds)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

vila_real_4326.plot(
    ax=ax,
    edgecolor="black",
    facecolor="none",
    linewidth=1.5
)

ax.set_title("Município de Vila Real — CAOP2025")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()

In [ ]:
import numpy as np
from rasterio.mask import mask

# Geometria do município no mesmo CRS do CHELSA
geometry = [
    vila_real_4326.geometry.iloc[0].__geo_interface__
]

with rasterio.open(output_path) as src:
    # Recorta o raster usando o limite de Vila Real
    vila_real_raster, vila_real_transform = mask(
        src,
        geometry,
        crop=True,
        filled=False
    )

    scale = src.scales[0]
    offset = src.offsets[0]

print("Dimensão do raster global:")
with rasterio.open(output_path) as src:
    print(src.height, "x", src.width)

print("\nDimensão após recortar Vila Real:")
print(vila_real_raster.shape)

print("\nQuantidade de pixels válidos:")
print(vila_real_raster.count())

In [ ]:
vila_real_celsius = (
    vila_real_raster.astype("float32") * scale
    + offset
    - 273.15
)

print("Temperatura mínima nos pixels válidos:")
print(f"{vila_real_celsius.min():.2f} °C")

print("\nTemperatura máxima nos pixels válidos:")
print(f"{vila_real_celsius.max():.2f} °C")

In [ ]:
from rasterio.plot import plotting_extent

# Pegamos apenas a primeira banda
temperature_data = vila_real_celsius[0]

# Calcula a extensão geográfica correta do raster recortado
extent = plotting_extent(
    temperature_data,
    vila_real_transform
)

fig, ax = plt.subplots(figsize=(8, 7))

img = ax.imshow(
    temperature_data,
    extent=extent,
    origin="upper"
)

# Desenha o limite municipal por cima
vila_real_4326.boundary.plot(
    ax=ax,
    edgecolor="black",
    linewidth=1.5
)

# Barra de temperatura
cbar = fig.colorbar(img, ax=ax)
cbar.set_label("Temperatura média de janeiro (°C)")

ax.set_title(
    "Temperatura média climatológica de janeiro\n"
    "Vila Real — CHELSA 1981–2010"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()

In [ ]:
from exactextract import exact_extract

resultado = exact_extract(
    output_path,
    vila_real_4326,
    [
        "mean_kelvin=mean(coverage_weight=area_spherical_m2)",
        "covered_area_m2=count(coverage_weight=area_spherical_m2)"
    ],
    include_cols=["municipio"],
    output="pandas"
)

resultado

In [ ]:
mean_kelvin = resultado.loc[0, "mean_kelvin"]
mean_celsius = mean_kelvin - 273.15

area_km2 = resultado.loc[0, "covered_area_m2"] / 1_000_000
area_ha = resultado.loc[0, "covered_area_m2"] / 10_000

print(f"Temperatura média: {mean_kelvin:.2f} K")
print(f"Temperatura média: {mean_celsius:.2f} °C")
print(f"Área considerada: {area_km2:.2f} km²")
print(f"Área considerada: {area_ha:.2f} ha")
print(f"Área oficial CAOP: {vila_real.iloc[0]['area_ha']:.2f} ha")

In [ ]:
from urllib.request import urlretrieve

for month in range(1, 13):
    month_str = f"{month:02d}"

    filename = f"CHELSA_tas_{month_str}_1981-2010_V.2.1.tif"

    url = (
        "https://os.unil.cloud.switch.ch/"
        "chelsa02/chelsa/global/climatologies/tas/1981-2010/"
        f"{filename}"
    )

    file_path = CHELSA_DIR / filename

    if file_path.exists():
        print(f"{month_str}: já existe")
    else:
        print(f"{month_str}: baixando...")
        urlretrieve(url, file_path)
        print(f"{month_str}: concluído")

print("\nTodos os meses disponíveis.")

In [ ]:
monthly_results = []

for month in range(1, 13):
    month_str = f"{month:02d}"

    file_path = (
        CHELSA_DIR /
        f"CHELSA_tas_{month_str}_1981-2010_V.2.1.tif"
    )

    result = exact_extract(
        file_path,
        vila_real_4326,
        ["mean_kelvin=mean(coverage_weight=area_spherical_m2)"],
        output="pandas"
    )

    mean_kelvin = result.loc[0, "mean_kelvin"]
    mean_celsius = mean_kelvin - 273.15

    monthly_results.append({
        "month": month,
        "mean_kelvin": mean_kelvin,
        "mean_celsius": mean_celsius
    })

monthly_df = pd.DataFrame(monthly_results)

monthly_df

In [ ]:
month_days = {
    1: 31,
    2: 28,
    3: 31,
    4: 30,
    5: 31,
    6: 30,
    7: 31,
    8: 31,
    9: 30,
    10: 31,
    11: 30,
    12: 31
}

monthly_df["days"] = monthly_df["month"].map(month_days)

annual_mean_celsius = (
    (monthly_df["mean_celsius"] * monthly_df["days"]).sum()
    / monthly_df["days"].sum()
)

print(f"Temperatura média anual climatológica: {annual_mean_celsius:.2f} °C")

In [ ]:
month_names = [
    "Jan", "Fev", "Mar", "Abr", "Mai", "Jun",
    "Jul", "Ago", "Set", "Out", "Nov", "Dez"
]

monthly_df["month_name"] = month_names

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    monthly_df["month_name"],
    monthly_df["mean_celsius"],
    marker="o"
)

ax.axhline(
    annual_mean_celsius,
    linestyle="--",
    label=f"Média anual: {annual_mean_celsius:.2f} °C"
)

ax.set_title(
    "Temperatura média climatológica mensal\n"
    "Vila Real — CHELSA 1981–2010"
)

ax.set_xlabel("Mês")
ax.set_ylabel("Temperatura média (°C)")
ax.legend()
ax.grid(alpha=0.3)

plt.show()

In [ ]:
import calendar

years = range(1981, 2011)

days_by_month = {
    month: sum(
        calendar.monthrange(year, month)[1]
        for year in years
    )
    for month in range(1, 13)
}

days_by_month

In [ ]:
monthly_df["days_1981_2010"] = (
    monthly_df["month"].map(days_by_month)
)

annual_mean_exact = (
    (
        monthly_df["mean_celsius"]
        * monthly_df["days_1981_2010"]
    ).sum()
    / monthly_df["days_1981_2010"].sum()
)

print(
    "Temperatura média anual climatológica "
    f"(ponderação exata 1981–2010): "
    f"{annual_mean_exact:.4f} °C"
)

print(
    "Diferença em relação ao cálculo anterior:",
    f"{annual_mean_exact - annual_mean_celsius:.4f} °C"
)

In [ ]:
# Tabela mensal final
monthly_output = monthly_df[
    ["month", "month_name", "mean_kelvin", "mean_celsius", "days_1981_2010"]
].copy()

monthly_output["municipality"] = "Vila Real"
monthly_output["variable"] = "tas"
monthly_output["period"] = "1981-2010"
monthly_output["source"] = "CHELSA climatologies v2.1"

monthly_output

In [ ]:
monthly_output_path = (
    PROCESSED_DIR /
    "vila_real_tas_monthly_1981_2010.parquet"
)

monthly_output.to_parquet(
    monthly_output_path,
    index=False
)

print("Salvo em:")
print(monthly_output_path)

In [ ]:
annual_output = pd.DataFrame([
    {
        "municipality": "Vila Real",
        "variable": "tas",
        "period": "1981-2010",
        "mean_celsius": annual_mean_exact,
        "source": "CHELSA climatologies v2.1"
    }
])

annual_output

In [ ]:
annual_output_path = (
    PROCESSED_DIR /
    "vila_real_tas_annual_1981_2010.parquet"
)

annual_output.to_parquet(
    annual_output_path,
    index=False
)

print("Salvo em:")
print(annual_output_path)

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.climate_processing import calculate_monthly_temperature

print("Função importada com sucesso!")

In [ ]:
january_raster = (
    CHELSA_DIR /
    "CHELSA_tas_01_1981-2010_V.2.1.tif"
)

january_result = calculate_monthly_temperature(
    raster_path=january_raster,
    region_gdf=vila_real_4326,
    month=1,
    municipality_name="Vila Real"
)

january_result

In [ ]:
expected_january = monthly_df.loc[
    monthly_df["month"] == 1,
    "mean_celsius"
].iloc[0]

difference = abs(
    january_result["mean_celsius"] - expected_january
)

print(f"Resultado da função: {january_result['mean_celsius']:.6f} °C")
print(f"Resultado anterior: {expected_january:.6f} °C")
print(f"Diferença: {difference:.10f} °C")

In [ ]:
import importlib
import src.climate_processing as cp

importlib.reload(cp)

In [ ]:
monthly_test = cp.calculate_monthly_temperature_climatology(
    chelsa_dir=CHELSA_DIR,
    region_gdf=vila_real_4326,
    municipality_name="Vila Real",
    period="1981-2010",
)

monthly_test

In [ ]:
comparison = monthly_test.merge(
    monthly_df[["month", "mean_celsius"]],
    on="month",
    suffixes=("_function", "_original")
)

comparison["difference"] = abs(
    comparison["mean_celsius_function"]
    - comparison["mean_celsius_original"]
)

comparison

In [ ]:
print(
    "Maior diferença:",
    comparison["difference"].max()
)

In [ ]:
annual_test = cp.calculate_annual_temperature_climatology(
    monthly_df=monthly_test,
    start_year=1981,
    end_year=2010,
)

print(
    f"Média anual pela função: "
    f"{annual_test:.4f} °C"
)

In [ ]:
difference_annual = abs(
    annual_test - annual_mean_exact
)

print(
    f"Resultado da função: {annual_test:.6f} °C"
)

print(
    f"Resultado anterior: {annual_mean_exact:.6f} °C"
)

print(
    f"Diferença: {difference_annual:.10f} °C"
)

In [ ]:
monthly_pipeline, annual_pipeline = (
    cp.process_temperature_climatology(
        chelsa_dir=CHELSA_DIR,
        region_gdf=vila_real_4326,
        municipality_name="Vila Real",
        period="1981-2010",
    )
)

In [ ]:
monthly_pipeline

In [ ]:
annual_pipeline

In [ ]:
print(
    "Média anual:",
    annual_pipeline.loc[0, "mean_celsius"]
)

print(
    "Número de meses:",
    len(monthly_pipeline)
)

print(
    "Município:",
    annual_pipeline.loc[0, "municipality"]
)

print(
    "Período:",
    annual_pipeline.loc[0, "period"]
)

In [ ]:
import src.data_io as dio

importlib.reload(dio)

In [ ]:
monthly_path, annual_path = dio.save_climatology_results(
    monthly_df=monthly_pipeline,
    annual_df=annual_pipeline,
    output_dir=PROCESSED_DIR,
    municipality_slug="vila_real",
    variable="tas",
    period="1981-2010",
)

print("Mensal:")
print(monthly_path)

print("\nAnual:")
print(annual_path)

In [ ]:
monthly_loaded = pd.read_parquet(monthly_path)
annual_loaded = pd.read_parquet(annual_path)

print("Mensal:")
display(monthly_loaded.head())

print("\nAnual:")
display(annual_loaded)

In [ ]:
assert len(monthly_loaded) == 12

assert abs(
    annual_loaded.loc[0, "mean_celsius"]
    - 12.029871426813624
) < 1e-10

print("Persistência validada com sucesso!")

In [ ]:
braganca = municipios[
    municipios["municipio"].str.strip().str.casefold()
    == "bragança".casefold()
].copy()

print(braganca)

print("\nQuantidade de registros encontrados:", len(braganca))

In [ ]:
print("Município:", braganca.iloc[0]["municipio"])
print("Distrito:", braganca.iloc[0]["distrito_ilha"])
print("NUTS III:", braganca.iloc[0]["nuts3"])
print("NUTS II:", braganca.iloc[0]["nuts2"])
print("Área oficial (ha):", braganca.iloc[0]["area_ha"])
print("Número de freguesias:", braganca.iloc[0]["n_freguesias"])
print("CRS:", braganca.crs)

In [ ]:
braganca_4326 = braganca.to_crs(epsg=4326)

print("\nCRS reprojetado:", braganca_4326.crs)
print("Bounds:", braganca_4326.total_bounds)

In [ ]:
monthly_braganca, annual_braganca = (
    cp.process_temperature_climatology(
        chelsa_dir=CHELSA_DIR,
        region_gdf=braganca_4326,
        municipality_name="Bragança",
        period="1981-2010",
    )
)

In [ ]:
monthly_braganca

In [ ]:
annual_braganca

In [ ]:
print(
    "Média anual de Bragança:",
    f"{annual_braganca.loc[0, 'mean_celsius']:.4f} °C"
)

print(
    "Número de meses:",
    len(monthly_braganca)
)

print(
    "Município:",
    annual_braganca.loc[0, "municipality"]
)

print(
    "Período:",
    annual_braganca.loc[0, "period"]
)

In [ ]:
monthly_braganca_path, annual_braganca_path = (
    dio.save_climatology_results(
        monthly_df=monthly_braganca,
        annual_df=annual_braganca,
        output_dir=PROCESSED_DIR,
        municipality_slug="braganca",
        variable="tas",
        period="1981-2010",
    )
)

print("Mensal:")
print(monthly_braganca_path)

print("\nAnual:")
print(annual_braganca_path)

In [ ]:
monthly_braganca_loaded = pd.read_parquet(
    monthly_braganca_path
)

annual_braganca_loaded = pd.read_parquet(
    annual_braganca_path
)

assert len(monthly_braganca_loaded) == 12

assert abs(
    annual_braganca_loaded.loc[0, "mean_celsius"]
    - annual_braganca.loc[0, "mean_celsius"]
) < 1e-10

print("Bragança persistida e validada com sucesso!")

In [ ]:
monthly_comparison = (
    monthly_pipeline[
        ["month", "mean_celsius"]
    ]
    .rename(
        columns={
            "mean_celsius": "vila_real_celsius"
        }
    )
    .merge(
        monthly_braganca[
            ["month", "mean_celsius"]
        ].rename(
            columns={
                "mean_celsius": "braganca_celsius"
            }
        ),
        on="month",
    )
)

monthly_comparison[
    "difference_braganca_minus_vila_real"
] = (
    monthly_comparison["braganca_celsius"]
    - monthly_comparison["vila_real_celsius"]
)

monthly_comparison

In [ ]:
print(
    "Menor diferença mensal:",
    monthly_comparison[
        "difference_braganca_minus_vila_real"
    ].min()
)

print(
    "Maior diferença mensal:",
    monthly_comparison[
        "difference_braganca_minus_vila_real"
    ].max()
)

In [ ]:
import matplotlib.pyplot as plt

month_labels = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    monthly_comparison["month"],
    monthly_comparison["vila_real_celsius"],
    marker="o",
    label="Vila Real"
)

ax.plot(
    monthly_comparison["month"],
    monthly_comparison["braganca_celsius"],
    marker="o",
    label="Bragança"
)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)

ax.set_xlabel("Month")
ax.set_ylabel("Mean temperature (°C)")

ax.set_title(
    "Monthly Mean Temperature Climatology — 1981–2010"
)

ax.grid(
    alpha=0.3
)

ax.legend()

figure_path = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
    / "vila_real_vs_braganca_tas_1981-2010.png"
)

plt.tight_layout()

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

print("Figura salva em:")
print(figure_path)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(
    monthly_comparison["month"],
    monthly_comparison[
        "difference_braganca_minus_vila_real"
    ]
)

ax.axhline(
    0,
    linewidth=1
)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)

ax.set_xlabel("Month")
ax.set_ylabel("Temperature difference (°C)")

ax.set_title(
    "Bragança − Vila Real Monthly Temperature Difference — 1981–2010"
)

ax.grid(
    axis="y",
    alpha=0.3
)

difference_figure_path = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
    / "braganca_minus_vila_real_tas_1981-2010.png"
)

plt.tight_layout()

plt.savefig(
    difference_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Figura salva em:")
print(difference_figure_path)

plt.show()

In [ ]:
import src.climate_analysis as ca

importlib.reload(ca)

In [ ]:
comparison_test = ca.compare_monthly_temperature(
    first_df=monthly_pipeline,
    second_df=monthly_braganca,
)

comparison_test

In [ ]:
difference_test = abs(
    comparison_test["difference_celsius"]
    - monthly_comparison[
        "difference_braganca_minus_vila_real"
    ]
).max()

print(
    f"Maior diferença entre os métodos: "
    f"{difference_test:.10f} °C"
)

In [ ]:
comparison_v2 = ca.compare_monthly_temperature(
    first_df=monthly_pipeline,
    second_df=monthly_braganca
)

comparison_v2

In [ ]:
difference_validation = abs(
    comparison_v2["difference_celsius"]
    - monthly_comparison[
        "difference_braganca_minus_vila_real"
    ]
).max()

print(
    f"Maior diferença após refatoração: "
    f"{difference_validation:.10f} °C"
)

In [ ]:
comparison_v3 = ca.compare_monthly_temperature(
    first_df=monthly_pipeline,
    second_df=monthly_braganca,
)

comparison_v3

In [ ]:
difference_validation_v3 = abs(
    comparison_v3["difference_celsius"]
    - comparison_v2["difference_celsius"]
).max()

print(
    f"Maior diferença após automatizar os nomes: "
    f"{difference_validation_v3:.10f} °C"
)

print(
    "Primeira região:",
    comparison_v3["first_region"].iloc[0]
)

print(
    "Segunda região:",
    comparison_v3["second_region"].iloc[0]
)

In [ ]:
import src.boundary_processing as bp

importlib.reload(bp)

In [ ]:
braganca_test = bp.get_municipality_geometry(
    municipalities_gdf=municipios,
    municipality_name="Bragança",
)

print("Município:", braganca_test.iloc[0]["municipio"])
print("CRS:", braganca_test.crs)
print("Área oficial (ha):", braganca_test.iloc[0]["area_ha"])
print("Bounds:", braganca_test.total_bounds)

In [ ]:
monthly_braganca_auto, annual_braganca_auto = (
    cp.process_temperature_climatology(
        chelsa_dir=CHELSA_DIR,
        region_gdf=braganca_test,
        municipality_name="Bragança",
        period="1981-2010",
    )
)

In [ ]:
annual_difference = abs(
    annual_braganca_auto.loc[0, "mean_celsius"]
    - annual_braganca.loc[0, "mean_celsius"]
)

print(
    "Resultado com geometria automática:",
    f"{annual_braganca_auto.loc[0, 'mean_celsius']:.10f} °C"
)

print(
    "Resultado anterior:",
    f"{annual_braganca.loc[0, 'mean_celsius']:.10f} °C"
)

print(
    "Diferença:",
    f"{annual_difference:.10f} °C"
)

In [ ]:
monthly_difference = abs(
    monthly_braganca_auto["mean_celsius"]
    - monthly_braganca["mean_celsius"]
).max()

print(
    "Maior diferença mensal:",
    f"{monthly_difference:.10f} °C"
)

In [ ]:
import src.climate_pipeline as pipeline

importlib.reload(pipeline)

In [ ]:
monthly_braganca_pipeline, annual_braganca_pipeline = (
    pipeline.process_municipality_temperature(
        municipalities_gdf=municipios,
        municipality_name="Bragança",
        chelsa_dir=CHELSA_DIR,
        period="1981-2010",
    )
)

In [ ]:
annual_pipeline_difference = abs(
    annual_braganca_pipeline.loc[0, "mean_celsius"]
    - annual_braganca.loc[0, "mean_celsius"]
)

monthly_pipeline_difference = abs(
    monthly_braganca_pipeline["mean_celsius"]
    - monthly_braganca["mean_celsius"]
).max()

print(
    "Diferença anual:",
    f"{annual_pipeline_difference:.10f} °C"
)

print(
    "Maior diferença mensal:",
    f"{monthly_pipeline_difference:.10f} °C"
)

In [ ]:
monthly_chaves, annual_chaves = (
    pipeline.process_municipality_temperature(
        municipalities_gdf=municipios,
        municipality_name="Chaves",
        chelsa_dir=CHELSA_DIR,
        period="1981-2010",
    )
)

In [ ]:
annual_chaves

In [ ]:
print(
    "Média anual de Chaves:",
    f"{annual_chaves.loc[0, 'mean_celsius']:.4f} °C"
)

print(
    "Número de meses:",
    len(monthly_chaves)
)

print(
    "Município:",
    annual_chaves.loc[0, "municipality"]
)

print(
    "Período:",
    annual_chaves.loc[0, "period"]
)

In [ ]:
monthly_chaves_path, annual_chaves_path = (
    dio.save_climatology_results(
        monthly_df=monthly_chaves,
        annual_df=annual_chaves,
        output_dir=PROCESSED_DIR,
        municipality_slug="chaves",
        variable="tas",
        period="1981-2010",
    )
)

print("Mensal:")
print(monthly_chaves_path)

print("\nAnual:")
print(annual_chaves_path)

In [ ]:
monthly_chaves_loaded = pd.read_parquet(
    monthly_chaves_path
)

annual_chaves_loaded = pd.read_parquet(
    annual_chaves_path
)

assert len(monthly_chaves_loaded) == 12

assert abs(
    annual_chaves_loaded.loc[0, "mean_celsius"]
    - annual_chaves.loc[0, "mean_celsius"]
) < 1e-10

print("Chaves persistida e validada com sucesso!")

In [ ]:
import importlib
import src.climate_pipeline as pipeline

importlib.reload(pipeline)

In [ ]:
municipalities_test = [
    "Vila Real",
    "Bragança",
    "Chaves",
]

monthly_batch, annual_batch = (
    pipeline.process_multiple_municipalities_temperature(
        municipalities_gdf=municipios,
        municipality_names=municipalities_test,
        chelsa_dir=CHELSA_DIR,
        period="1981-2010",
    )
)

In [ ]:
annual_batch

In [ ]:
print(
    "Registros mensais:",
    len(monthly_batch)
)

print(
    "Registros anuais:",
    len(annual_batch)
)

print(
    "Municípios:",
    annual_batch["municipality"].tolist()
)

In [ ]:
print(
    monthly_batch
    .groupby("municipality")
    .size()
)

In [ ]:
monthly_individual = pd.concat(
    [
        monthly_pipeline,
        monthly_braganca,
        monthly_chaves,
    ],
    ignore_index=True,
)

annual_individual = pd.concat(
    [
        annual_pipeline,
        annual_braganca,
        annual_chaves,
    ],
    ignore_index=True,
)

In [ ]:
monthly_validation = monthly_batch.merge(
    monthly_individual[
        [
            "municipality",
            "month",
            "mean_celsius",
        ]
    ],
    on=[
        "municipality",
        "month",
    ],
    suffixes=(
        "_batch",
        "_individual",
    ),
)

monthly_validation["difference"] = abs(
    monthly_validation["mean_celsius_batch"]
    - monthly_validation["mean_celsius_individual"]
)

print(
    "Maior diferença mensal:",
    f"{monthly_validation['difference'].max():.10f} °C"
)

In [ ]:
annual_validation = annual_batch.merge(
    annual_individual[
        [
            "municipality",
            "mean_celsius",
        ]
    ],
    on="municipality",
    suffixes=(
        "_batch",
        "_individual",
    ),
)

annual_validation["difference"] = abs(
    annual_validation["mean_celsius_batch"]
    - annual_validation["mean_celsius_individual"]
)

print(
    "Maior diferença anual:",
    f"{annual_validation['difference'].max():.10f} °C"
)

In [ ]:
import importlib
import src.data_io as dio

importlib.reload(dio)

In [ ]:
batch_monthly_path, batch_annual_path = (
    dio.save_batch_climatology_results(
        monthly_df=monthly_batch,
        annual_df=annual_batch,
        output_dir=PROCESSED_DIR,
        variable="tas",
        period="1981-2010",
    )
)

print("Mensal consolidado:")
print(batch_monthly_path)

print("\nAnual consolidado:")
print(batch_annual_path)

In [ ]:
monthly_batch_loaded = pd.read_parquet(
    batch_monthly_path
)

annual_batch_loaded = pd.read_parquet(
    batch_annual_path
)

assert len(monthly_batch_loaded) == 36
assert len(annual_batch_loaded) == 3

assert set(
    annual_batch_loaded["municipality"]
) == {
    "Vila Real",
    "Bragança",
    "Chaves",
}

print(
    "Resultados consolidados persistidos "
    "e validados com sucesso!"
)

In [ ]:
import importlib
import src.boundary_processing as bp

importlib.reload(bp)

In [ ]:
douro_municipalities = (
    bp.get_municipality_names_by_nuts3(
        municipalities_gdf=municipios,
        nuts3_name="Douro",
    )
)

print(
    "Quantidade de municípios:",
    len(douro_municipalities)
)

print("\nMunicípios da NUTS III Douro:")

for municipality in douro_municipalities:
    print("-", municipality)

In [ ]:
print(
    "\nVila Real está na lista?",
    "Vila Real" in douro_municipalities
)

In [ ]:
douro_gdf = bp.get_municipalities_by_nuts3(
    municipalities_gdf=municipios,
    nuts3_name="Douro",
)

print("Quantidade:", len(douro_gdf))
print("CRS:", douro_gdf.crs)

display(
    douro_gdf[
        [
            "municipio",
            "nuts3",
            "area_ha",
        ]
    ]
)

In [ ]:
import importlib
import src.climate_processing as cp

importlib.reload(cp)

In [ ]:
january_raster = (
    CHELSA_DIR
    / "CHELSA_tas_01_1981-2010_V.2.1.tif"
)

In [ ]:
douro_january = (
    cp.calculate_monthly_temperature_for_regions(
        raster_path=january_raster,
        regions_gdf=douro_gdf,
        month=1,
    )
)

douro_january

In [ ]:
print(
    "Número de municípios processados:",
    len(douro_january)
)

print(
    "Municípios únicos:",
    douro_january["municipality"].nunique()
)

In [ ]:
vila_real_january_regional = (
    douro_january.loc[
        douro_january["municipality"] == "Vila Real",
        "mean_celsius",
    ]
    .iloc[0]
)

vila_real_january_individual = (
    monthly_pipeline.loc[
        monthly_pipeline["month"] == 1,
        "mean_celsius",
    ]
    .iloc[0]
)

difference = abs(
    vila_real_january_regional
    - vila_real_january_individual
)

print(
    "Vila Real - processamento regional:",
    f"{vila_real_january_regional:.10f} °C"
)

print(
    "Vila Real - processamento individual:",
    f"{vila_real_january_individual:.10f} °C"
)

print(
    "Diferença:",
    f"{difference:.10f} °C"
)

In [ ]:
douro_monthly = (
    cp.calculate_monthly_temperature_climatology_for_regions(
        chelsa_dir=CHELSA_DIR,
        regions_gdf=douro_gdf,
        period="1981-2010",
    )
)

In [ ]:
print(
    "Total de registros:",
    len(douro_monthly)
)

print(
    "Municípios:",
    douro_monthly["municipality"].nunique()
)

print(
    "Meses:",
    douro_monthly["month"].nunique()
)

In [ ]:
months_per_municipality = (
    douro_monthly
    .groupby("municipality")
    .size()
)

print(months_per_municipality)

print(
    "\nTodos possuem 12 meses?",
    (months_per_municipality == 12).all()
)

In [ ]:
vila_real_regional = (
    douro_monthly[
        douro_monthly["municipality"] == "Vila Real"
    ]
    .sort_values("month")
    .reset_index(drop=True)
)

vila_real_individual = (
    monthly_pipeline
    .sort_values("month")
    .reset_index(drop=True)
)

vila_real_validation = abs(
    vila_real_regional["mean_celsius"]
    - vila_real_individual["mean_celsius"]
)

print(
    "Maior diferença entre regional e individual:",
    f"{vila_real_validation.max():.10f} °C"
)

In [ ]:
douro_annual = (
    cp.calculate_annual_temperature_climatology_for_regions(
        monthly_df=douro_monthly,
        start_year=1981,
        end_year=2010,
    )
)

douro_annual

In [ ]:
print(
    "Municípios anuais:",
    len(douro_annual)
)

print(
    "Municípios únicos:",
    douro_annual["municipality"].nunique()
)

In [ ]:
vila_real_annual_regional = (
    douro_annual.loc[
        douro_annual["municipality"] == "Vila Real",
        "mean_celsius",
    ]
    .iloc[0]
)

vila_real_annual_individual = (
    annual_pipeline.loc[
        0,
        "mean_celsius",
    ]
)

difference = abs(
    vila_real_annual_regional
    - vila_real_annual_individual
)

print(
    "Vila Real regional:",
    f"{vila_real_annual_regional:.10f} °C"
)

print(
    "Vila Real individual:",
    f"{vila_real_annual_individual:.10f} °C"
)

print(
    "Diferença:",
    f"{difference:.10f} °C"
)

In [ ]:
import importlib
import src.climate_processing as cp

importlib.reload(cp)

In [ ]:
douro_monthly_pipeline, douro_annual_pipeline = (
    cp.process_temperature_climatology_for_regions(
        chelsa_dir=CHELSA_DIR,
        regions_gdf=douro_gdf,
        period="1981-2010",
    )
)

In [ ]:
print(
    "Registros mensais:",
    len(douro_monthly_pipeline)
)

print(
    "Registros anuais:",
    len(douro_annual_pipeline)
)

print(
    "Municípios:",
    douro_annual_pipeline["municipality"].nunique()
)

display(
    douro_annual_pipeline.head()
)

In [ ]:
monthly_difference = abs(
    douro_monthly_pipeline["mean_celsius"]
    - douro_monthly["mean_celsius"]
).max()

annual_check = (
    douro_annual_pipeline[
        ["municipality", "mean_celsius"]
    ]
    .merge(
        douro_annual,
        on="municipality",
        suffixes=("_pipeline", "_previous"),
    )
)

annual_difference = abs(
    annual_check["mean_celsius_pipeline"]
    - annual_check["mean_celsius_previous"]
).max()

print(
    "Maior diferença mensal:",
    f"{monthly_difference:.10f} °C"
)

print(
    "Maior diferença anual:",
    f"{annual_difference:.10f} °C"
)

In [ ]:
import importlib
import src.climate_pipeline as pipeline

importlib.reload(pipeline)

In [ ]:
douro_monthly_auto, douro_annual_auto = (
    pipeline.process_nuts3_temperature(
        municipalities_gdf=municipios,
        nuts3_name="Douro",
        chelsa_dir=CHELSA_DIR,
        period="1981-2010",
    )
)

In [ ]:
print(
    "Registros mensais:",
    len(douro_monthly_auto)
)

print(
    "Registros anuais:",
    len(douro_annual_auto)
)

print(
    "Municípios:",
    douro_annual_auto["municipality"].nunique()
)

In [ ]:
monthly_difference = abs(
    douro_monthly_auto["mean_celsius"]
    - douro_monthly_pipeline["mean_celsius"]
).max()

annual_validation = (
    douro_annual_auto[
        ["municipality", "mean_celsius"]
    ]
    .merge(
        douro_annual_pipeline[
            ["municipality", "mean_celsius"]
        ],
        on="municipality",
        suffixes=("_auto", "_previous"),
    )
)

annual_difference = abs(
    annual_validation["mean_celsius_auto"]
    - annual_validation["mean_celsius_previous"]
).max()

print(
    "Maior diferença mensal:",
    f"{monthly_difference:.10f} °C"
)

print(
    "Maior diferença anual:",
    f"{annual_difference:.10f} °C"
)

In [ ]:
import importlib
import src.data_io as dio

importlib.reload(dio)

In [ ]:
douro_monthly_path, douro_annual_path = (
    dio.save_regional_climatology_results(
        monthly_df=douro_monthly_auto,
        annual_df=douro_annual_auto,
        output_dir=PROCESSED_DIR,
        region_slug="douro",
        variable="tas",
        period="1981-2010",
    )
)

print("Mensal regional:")
print(douro_monthly_path)

print("\nAnual regional:")
print(douro_annual_path)

In [ ]:
douro_monthly_loaded = pd.read_parquet(
    douro_monthly_path
)

douro_annual_loaded = pd.read_parquet(
    douro_annual_path
)

assert len(douro_monthly_loaded) == 228
assert len(douro_annual_loaded) == 19

assert (
    douro_monthly_loaded["municipality"].nunique()
    == 19
)

assert (
    douro_annual_loaded["municipality"].nunique()
    == 19
)

print(
    "Douro persistido e validado com sucesso!"
)

In [ ]:
import importlib
import src.boundary_processing as bp

importlib.reload(bp)

In [ ]:
three_municipalities = (
    bp.get_municipalities_by_names(
        municipalities_gdf=municipios,
        municipality_names=[
            "Vila Real",
            "Bragança",
            "Chaves",
        ],
    )
)

print(
    three_municipalities["municipio"].tolist()
)

print(
    "Quantidade:",
    len(three_municipalities)
)

print(
    "CRS:",
    three_municipalities.crs
)

In [ ]:
import importlib
import src.climate_pipeline as pipeline

importlib.reload(pipeline)

In [ ]:
monthly_batch_v2, annual_batch_v2 = (
    pipeline.process_multiple_municipalities_temperature(
        municipalities_gdf=municipios,
        municipality_names=[
            "Vila Real",
            "Bragança",
            "Chaves",
        ],
        chelsa_dir=CHELSA_DIR,
        period="1981-2010",
    )
)

In [ ]:
monthly_check = (
    monthly_batch_v2[
        ["municipality", "month", "mean_celsius"]
    ]
    .merge(
        monthly_batch[
            ["municipality", "month", "mean_celsius"]
        ],
        on=[
            "municipality",
            "month",
        ],
        suffixes=(
            "_new",
            "_old",
        ),
    )
)

monthly_difference = abs(
    monthly_check["mean_celsius_new"]
    - monthly_check["mean_celsius_old"]
).max()


annual_check = (
    annual_batch_v2[
        ["municipality", "mean_celsius"]
    ]
    .merge(
        annual_batch[
            ["municipality", "mean_celsius"]
        ],
        on="municipality",
        suffixes=(
            "_new",
            "_old",
        ),
    )
)

annual_difference = abs(
    annual_check["mean_celsius_new"]
    - annual_check["mean_celsius_old"]
).max()


print(
    "Maior diferença mensal:",
    f"{monthly_difference:.10f} °C"
)

print(
    "Maior diferença anual:",
    f"{annual_difference:.10f} °C"
)

In [ ]:
import importlib
import src.data_io as dio

importlib.reload(dio)

In [ ]:
test_monthly_path, test_annual_path = (
    dio.save_climatology_results(
        monthly_df=monthly_pipeline,
        annual_df=annual_pipeline,
        output_dir=PROCESSED_DIR,
        municipality_slug="vila_real",
        variable="tas",
        period="1981-2010",
    )
)

print(test_monthly_path)
print(test_annual_path)

In [ ]:
test_monthly_loaded = pd.read_parquet(
    test_monthly_path
)

test_annual_loaded = pd.read_parquet(
    test_annual_path
)

assert len(test_monthly_loaded) == 12

assert abs(
    test_annual_loaded.loc[0, "mean_celsius"]
    - annual_pipeline.loc[0, "mean_celsius"]
) < 1e-10

print("Refatoração da persistência validada!")

In [ ]:
import importlib
import src.data_io as dio

importlib.reload(dio)

In [ ]:
batch_test_monthly_path, batch_test_annual_path = (
    dio.save_batch_climatology_results(
        monthly_df=monthly_batch_v2,
        annual_df=annual_batch_v2,
        output_dir=PROCESSED_DIR,
        variable="tas",
        period="1981-2010",
    )
)

print(batch_test_monthly_path)
print(batch_test_annual_path)

In [ ]:
regional_test_monthly_path, regional_test_annual_path = (
    dio.save_regional_climatology_results(
        monthly_df=douro_monthly_auto,
        annual_df=douro_annual_auto,
        output_dir=PROCESSED_DIR,
        region_slug="douro",
        variable="tas",
        period="1981-2010",
    )
)

print(regional_test_monthly_path)
print(regional_test_annual_path)

In [ ]:
assert len(pd.read_parquet(batch_test_monthly_path)) == 36
assert len(pd.read_parquet(batch_test_annual_path)) == 3

assert len(pd.read_parquet(regional_test_monthly_path)) == 228
assert len(pd.read_parquet(regional_test_annual_path)) == 19

print("Refatoração completa do data_io validada!")

In [ ]:
import importlib
import src.climate_processing as cp

importlib.reload(cp)

In [ ]:
monthly_wrapper_test, annual_wrapper_test = (
    cp.process_temperature_climatology(
        chelsa_dir=CHELSA_DIR,
        region_gdf=vila_real_4326,
        municipality_name="Vila Real",
        period="1981-2010",
    )
)

In [ ]:
monthly_difference = abs(
    monthly_wrapper_test["mean_celsius"]
    - monthly_pipeline["mean_celsius"]
).max()

annual_difference = abs(
    annual_wrapper_test.loc[0, "mean_celsius"]
    - annual_pipeline.loc[0, "mean_celsius"]
)

print(
    "Maior diferença mensal:",
    f"{monthly_difference:.10f} °C"
)

print(
    "Diferença anual:",
    f"{annual_difference:.10f} °C"
)

In [ ]:
import importlib
import src.climate_processing as cp

importlib.reload(cp)

In [ ]:
monthly_wrapper_v2 = (
    cp.calculate_monthly_temperature_climatology(
        chelsa_dir=CHELSA_DIR,
        region_gdf=vila_real_4326,
        municipality_name="Vila Real",
        period="1981-2010",
    )
)

monthly_wrapper_v2

In [ ]:
monthly_wrapper_difference = abs(
    monthly_wrapper_v2["mean_celsius"]
    - monthly_test["mean_celsius"]
).max()

print(
    "Maior diferença:",
    f"{monthly_wrapper_difference:.10f} °C"
)

print(
    "Colunas:",
    monthly_wrapper_v2.columns.tolist()
)

In [ ]:
annual_wrapper_test = (
    cp.calculate_annual_temperature_climatology(
        monthly_df=monthly_test,
        start_year=1981,
        end_year=2010,
    )
)

print(
    "Resultado:",
    f"{annual_wrapper_test:.10f} °C"
)

In [ ]:
annual_wrapper_difference = abs(
    annual_wrapper_test
    - annual_mean_exact
)

print(
    "Diferença:",
    f"{annual_wrapper_difference:.10f} °C"
)

In [ ]:
import importlib
import src.climate_processing as cp

importlib.reload(cp)

In [ ]:
january_wrapper_test = cp.calculate_monthly_temperature(
    raster_path=january_raster,
    region_gdf=vila_real_4326,
    month=1,
    municipality_name="Vila Real",
)

january_wrapper_test

In [ ]:
january_difference = abs(
    january_wrapper_test["mean_celsius"]
    - january_result["mean_celsius"]
)

print(
    "Resultado novo:",
    f"{january_wrapper_test['mean_celsius']:.10f} °C"
)

print(
    "Resultado original:",
    f"{january_result['mean_celsius']:.10f} °C"
)

print(
    "Diferença:",
    f"{january_difference:.10f} °C"
)